# ODA-LAB extract SET-1

**Runtime → Run all.** Allow Drive.

Ten zips. SUNSET 416.6 MB is parked and not in this list.
Prefer a file already on MyDrive. Else gdown by id.
Dest: `12_ODA-LAB-NOTEBOOKS/extracts/SET1-<stamp>/`
Cap 50 MB. Originals stay put.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, zipfile, time, json, subprocess
print('=== mount ===')
drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive')
print('mounted', ROOT.exists())
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
print('gdown ok')


In [ ]:
import gdown
CAP = 50 * 1024 * 1024
PACKS = [
  [
    "STORYBOARD_RECON_20260910.zip",
    "1dYjyx6S1Tp3pWvPC6hchxe6OV5gQ69DX",
    2440000
  ],
  [
    "LIV_PANE_ARCHIVE_20260910.zip",
    "1518UAB_yra8VEvfTiRef-fEnZrfjZ5pB",
    37300000
  ],
  [
    "plate-pack-normalized-20260910.zip",
    "1G1pHWZuIRVEt9XIanZ6jmjeQAXJ3G7yE",
    4090000
  ],
  [
    "EMIT-20260910.zip",
    "1ZokwsXIfogBZfL2sP5du4n_q2Pnn9M5H",
    11400000
  ],
  [
    "ANIME-COPPER-20260910.zip",
    "1b7xEj1Wft8DJQzWH4GzQ_4TICnZNQM0i",
    1880000
  ],
  [
    "GRID-BLONDE-20260910.zip",
    "1yVCuq80O-eMI6E9OYJ_iXZfzCL05K0tG",
    2340000
  ],
  [
    "cand-reel-133985.zip",
    "12NkSa8VC_Qe13eqisQjuZiq_F3pefn8Y",
    10400000
  ],
  [
    "IP-WQ-161-166.zip",
    "18LTJ0HrF7DMuNpB0W2JMzZ1vaj1Rjd4T",
    4300
  ],
  [
    "IP-WQ-161-172.zip",
    "1t5Zal_pLKz5fwSANPnRjMy6YSbdMDl3W",
    12400
  ],
  [
    "OLIVIA_ORCA_SESSION_20260901.zip",
    "1uvLj8Gs0hluOPVVUukLNUqiLZYr9ppqa",
    36300000
  ]
]
PARKED = [('SUNSET_FULL_20260910_grok-five.tar.gz', '1q0yQh5Qoxp3Q-p2QmlpCCXe6dWQNzAO_')]

shelf = None
for cand in [ROOT / '12_ODA-LAB-NOTEBOOKS']:
    pass
hits = []
nwalk = 0
for dirpath, dirnames, filenames in os.walk(ROOT):
    nwalk += 1
    if '12_ODA-LAB-NOTEBOOKS' in dirnames and shelf is None:
        shelf = Path(dirpath) / '12_ODA-LAB-NOTEBOOKS'
    if nwalk > 4000:
        break
if shelf is None:
    shelf = ROOT / '12_ODA-LAB-NOTEBOOKS_LOCAL'
    shelf.mkdir(exist_ok=True)
dest_root = shelf / 'extracts' / ('SET1-' + time.strftime('%Y%m%d-%H%M'))
dest_root.mkdir(parents=True, exist_ok=True)
print('shelf', shelf)
print('dest', dest_root)

def find_name(name):
    n = 0
    for dirpath, dirnames, filenames in os.walk(ROOT):
        n += len(filenames)
        if name in filenames:
            return Path(dirpath) / name
        if n > 12000:
            break
    return None

lines = ['# ODA LAB extract SET-1', 'dest=' + str(dest_root), '']
for name, fid, claimed in PACKS:
    print('\n===', name, fid)
    src = find_name(name)
    if src is None:
        local = Path('/content') / name
        print('not on walk, gdown', fid)
        try:
            gdown.download(id=fid, output=str(local), quiet=False)
            src = local if local.is_file() else None
        except Exception as e:
            print('gdown fail', e)
            src = None
    if src is None or not src.is_file():
        lines.append('- FAIL missing ' + name + ' ' + fid)
        print('FAIL missing')
        continue
    sz = src.stat().st_size
    print('src', src, 'size', sz)
    if sz > CAP:
        lines.append('- SKIP cap %s %s' % (name, sz))
        print('SKIP cap')
        continue
    out = dest_root / src.stem
    if out.exists() and any(out.iterdir()):
        cnt = sum(1 for _ in out.rglob('*') if _.is_file())
        lines.append('- EXISTS %s files=%s' % (out, cnt))
        print('exists', cnt)
        continue
    out.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(src) as z:
            z.extractall(out)
            nlist = z.namelist()
        cnt = sum(1 for _ in out.rglob('*') if _.is_file())
        lines.append('- OK %s bytes=%s files=%s first=%s' % (name, sz, cnt, nlist[:6]))
        print('OK files', cnt)
    except Exception as e:
        lines.append('- FAIL unzip %s %s' % (name, e))
        print('FAIL unzip', e)

lines.append('')
lines.append('PARKED ' + str(PARKED))
receipt = dest_root / 'EXTRACT.RECEIPT.md'
receipt.write_text('\n'.join(str(x) for x in lines))
print('\nWROTE', receipt)
print('\n'.join(str(x) for x in lines))
